In [1]:
#!pip install ib_insync
#!pip install bs4
from ib_insync import *
from bs4 import BeautifulSoup as bs
ib = IB()

In [2]:
util.startLoop()
ib.disconnect()
ib.connect('127.0.0.1', 7497, clientId = 1)

<IB connected to 127.0.0.1:7497 clientId=1>

In [3]:
def onPendingTicker(ticker):
    print("pending ticker event received")
    print(ticker)
#ib.pendingTickersEvent += onPendingTicker

In [4]:
# Code for market scanning
subscription = ScannerSubscription(instrument = 'STK', locationCode = 'STK.US.MAJOR', scanCode = 'SCAN_currYrETFFYDividendYield_DESC')
scanData = ib.reqScannerData(subscription)
for scan in scanData:
    #print(scan)
    print(scan.contractDetails.contract.symbol)

DGRS
MORT
REM
KBWD
FDIV
MJUS
BIZD
RIET
SDIV
SRET
SMCP
KBWY
FYT
SQLV
SMMV
XSHD
RFV
RDOG
RFDA
AMLP
AMZA
DVYE
AFSM
OUSM
MLPA
DES
SDEM
EEMD
IDV
NETL
PEX
FGD
EFAS
EDOG
FDD
MJ
GXG
FLQS
NDIV
ECH
SEA
DIV
UMI
ENFR
DEM
DVYA
NORW
VWID
ENOR
WBIY


In [22]:
stock = Stock('INTC', 'SMART', 'USD')
bars = ib.reqHistoricalData(stock, endDateTime = '', durationStr = '30 D', barSizeSetting = '1 hour', whatToShow = 'MIDPOINT', useRTH = True, formatDate = 2)

In [23]:
def extract_info(accounts, tag):
    for acc in accounts:
        if tag == acc.tag:
            return acc.value
    return None

In [24]:
ib.qualifyContracts(stock)
account = ib.accountSummary()
print("Net worth:", extract_info(account, "NetLiquidation"))
print("Cash:", extract_info(account, "TotalCashValue"))

Net worth: 1030469.99
Cash: 1024455.02


In [25]:
print(stock)

Stock(conId=270639, symbol='INTC', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='INTC', tradingClass='NMS')


In [26]:
bars = ib.reqHistoricalData(stock, endDateTime = '', durationStr = '30 D', barSizeSetting = '1 hour', whatToShow = 'MIDPOINT', useRTH = True, formatDate = 2)
dfBars = util.df(bars)
dfBars

,date,open,high,low,close,volume,average,barCount
0,2024-04-17 13:30:00+00:00,36.09,36.14,35.77,35.98,-1.0,-1.0,-1
1,2024-04-17 14:00:00+00:00,35.98,35.98,35.81,35.89,-1.0,-1.0,-1
2,2024-04-17 15:00:00+00:00,35.89,36.00,35.65,35.70,-1.0,-1.0,-1
3,2024-04-17 16:00:00+00:00,35.70,35.71,35.37,35.45,-1.0,-1.0,-1
4,2024-04-17 17:00:00+00:00,35.45,35.82,35.43,35.80,-1.0,-1.0,-1
...,...,...,...,...,...,...,...,...
199,2024-05-28 16:00:00+00:00,31.29,31.32,31.06,31.07,-1.0,-1.0,-1
200,2024-05-28 17:00:00+00:00,31.07,31.18,31.03,31.05,-1.0,-1.0,-1
201,2024-05-28 18:00:00+00:00,31.05,31.09,30.78,30.88,-1.0,-1.0,-1
202,2024-05-28 19:00:00+00:00,30.88,31.11,30.82,31.09,-1.0,-1.0,-1


In [27]:
# * 'ReportsFinSummary': Financial summary
# * 'ReportsOwnership': Company's ownership
# * 'ReportSnapshot': Company's financial overview
# * 'ReportsFinStatements': Financial Statements
# * 'RESC': Analyst Estimates
# * 'CalendarReport': Company's calendar

fundamentals = ib.reqFundamentalData(stock, 'ReportSnapshot')

In [28]:
content = bs(fundamentals, "xml")
print(content)

<?xml version="1.0" encoding="utf-8"?>
<ReportSnapshot Major="1" Minor="0" Revision="1">
<CoIDs>
<CoID Type="RepNo">45870</CoID>
<CoID Type="CompanyName">Intel Corp</CoID>
<CoID Type="IRSNo">941672743</CoID>
<CoID Type="CIKNo">0000050863</CoID>
<CoID Type="OrganizationPermID">4295906830</CoID>
</CoIDs>
<Issues>
<Issue Desc="Common Stock" ID="1" Order="1" Type="C">
<IssueID Type="Name">Ordinary Shares</IssueID>
<IssueID Type="Ticker">INTC</IssueID>
<IssueID Type="RIC">INTC.O</IssueID>
<IssueID Type="DisplayRIC">INTC.OQ</IssueID>
<IssueID Type="InstrumentPI">261310</IssueID>
<IssueID Type="QuotePI">7737457</IssueID>
<IssueID Type="InstrumentPermID">8590927847</IssueID>
<IssueID Type="QuotePermID">55835340055</IssueID>
<Exchange Code="NASD" Country="USA">NASDAQ</Exchange>
<GlobalListingType>OSR</GlobalListingType>
<MostRecentSplit Date="2000-07-31">2.0</MostRecentSplit>
</Issue>
</Issues>
<CoGeneralInfo>
<CoStatus Code="1">Active</CoStatus>
<CoType Code="EQU">Equity Issue</CoType>
<LastMo

In [29]:
ratios = content.find_all("Ratio")
for ratio in ratios:
    print(ratio['FieldName'])
    print(ratio.text)

NPRICE
31.06000
NHIG
51.28000
NLOW
28.99500
PDATE
2024-05-28T00:00:00
VOL10DAVG
42.68744
EV
163361.40000
MKTCAP
132222.40000
TTMREV
55237.00000
TTMEBITD
11649.00000
TTMNIAC
4066.00000
TTMEPSXCLX
0.95292
TTMREVPS
13.01608
QBVPS
24.89382
QCSHPS
5.00611
TTMCFSHR
3.25019
TTMDIVSHR
0.50000
TTMGROSMGN
41.49393
TTMROEPCT
3.98565
TTMPR2REV
2.39373
PEEXCLXOR
32.59455
PRICE2BK
1.24770
Employees
124800
ConsRecom

2.7727

TargetPrice

38.32440

ProjLTGrowthRate

39.8400

ProjPE

28.31100

ProjSales

55816.51460

ProjSalesQ

12980.65700

ProjEPS

1.09710

ProjEPSQ

0.10290

ProjProfit

4583.36040

ProjDPS

0.57120



In [30]:
ib.sleep(0)
# Use this code to update the transmitted information

True

In [31]:
def orderFilled(trade, fill):
    print("order has been filled")
    print(order)
    print(fill)
#trade.fillEvent += orderFilled

In [37]:
# order = LimitOrder('Buy', 5, 91.33)
order = MarketOrder('BUY', 1)
trade = ib.placeOrder(stock, order)
# After running this code, order will be filled

In [38]:
print(trade)

Trade(contract=Stock(conId=270639, symbol='INTC', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='INTC', tradingClass='NMS'), order=MarketOrder(orderId=1300, clientId=1, permId=2122895804, action='BUY', totalQuantity=1.0, lmtPrice=0.0, auxPrice=0.0), orderStatus=OrderStatus(orderId=1300, status='Filled', filled=1.0, remaining=0.0, avgFillPrice=30.36, permId=2122895804, parentId=0, lastFillPrice=30.36, clientId=1, whyHeld='', mktCapPrice=0.0), fills=[Fill(contract=Stock(conId=270639, symbol='INTC', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='INTC', tradingClass='NMS'), execution=Execution(execId='00025b45.6656bf55.01.01', time=datetime.datetime(2024, 5, 29, 13, 36, 51, tzinfo=datetime.timezone.utc), acctNumber='DU8906537', exchange='EDGEA', side='BOT', shares=1.0, price=30.36, permId=2122895804, clientId=1, orderId=1300, liquidation=0, cumQty=1.0, avgPrice=30.36, orderRef='', evRule='', evMultiplier=0.0, modelCode='', lastLiquidity=2

In [39]:
def orderFilled(trade, fill):
    print("order has been filled")
    print(trade)
    print(fill)

trade.fillEvent += orderFilled

In [40]:
ib.sleep(3)

for trade in ib.trades():
    print("== this is one of my trades =")
    print(trade)

for order in ib.orders():
    print("== this is one of my orders ==")
    print(order)

ib.run()

== this is one of my trades =
Trade(contract=Stock(conId=107113386, symbol='META', right='?', exchange='SMART', currency='USD', localSymbol='META', tradingClass='NMS'), order=Order(permId=1278599637, action='SELL', orderType='MKT', lmtPrice=0.0, auxPrice=0.0, tif='DAY', ocaType=3, displaySize=2147483647, rule80A='0', openClose='', volatilityType=0, deltaNeutralOrderType='None', referencePriceType=0, account='DU8906537', clearingIntent='IB', cashQty=0.0, dontUseAutoPriceForHedge=True, filledQuantity=1.0, refFuturesConId=2147483647, shareholder='Not an insider or substantial shareholder'), orderStatus=OrderStatus(orderId=0, status='Filled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[Fill(contract=Stock(conId=107113386, symbol='META', right='?', exchange='SMART', currency='USD', localSymbol='META', tradingClass='NMS'), execution=Execution(execId='00025b45.6655d3f6.01.01', time=datetime.datetime(202

In [41]:
trade = ib.cancelOrder(order)

In [42]:
chains = ib.reqSecDefOptParams(stock.symbol, '', stock.secType, stock.conId)

In [43]:
dfChains = util.df(chains)
print(dfChains)

    exchange underlyingConId tradingClass multiplier  \
0      SMART          270639         INTC        100   
1       EDGX          270639         INTC        100   
2      CBOE2          270639         INTC        100   
3        ISE          270639         INTC        100   
4   NASDAQBX          270639         INTC        100   
5       MIAX          270639         INTC        100   
6       CBOE          270639         INTC        100   
7    MERCURY          270639         INTC        100   
8   NASDAQOM          270639         INTC        100   
9    IBUSOPT          270639         INTC        100   
10      BATS          270639         INTC        100   
11   EMERALD          270639         INTC        100   
12     PEARL          270639         INTC        100   
13      PHLX          270639         INTC        100   
14      AMEX          270639         INTC        100   
15    GEMINI          270639         INTC        100   
16       BOX          270639         INTC       